# Verify the risk-equivalent propagation probability

This notebook is a thin visualization of the tested package analysis. It selects the newest completed fully coupled `xyz` run matching the paper inputs and validates redundant sample-count and `pf_node` provenance before analysis. The numerical PMF equality, bounded crossing logic, Monte Carlo diagnostics, regular-lattice approximation, and comparison are implemented in `pbm.post`.

In [ ]:
from pathlib import Path
import sqlite3
import sys

CURRENT_DIR = Path.cwd().resolve()
REPO_ROOT = CURRENT_DIR if (CURRENT_DIR / 'pbm').is_dir() else CURRENT_DIR.parent
if not (REPO_ROOT / 'pbm').is_dir():
    raise RuntimeError('Run this notebook from the repository root or notebooks directory.')
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from pbm.post import select_latest_compatible_xyz_run
from pbm.post.core import run_postprocessing

OUT_DB = REPO_ROOT / 'output' / 'results_percolation.db'
FIGS = REPO_ROOT / 'output' / 'figs_percolation'
PE_CRIT = 0.67
LATTICE_SHAPE = (4, 4, 4)
EXPECTED_SAMPLES = 10_000_000
EXPECTED_PF_NODE = 1e-4
PLOT_MODE = 'standard'

FIGS.mkdir(parents=True, exist_ok=True)
print('OUT_DB =', OUT_DB)

In [ ]:
with sqlite3.connect(OUT_DB) as conn:
    selected_run = select_latest_compatible_xyz_run(
        conn,
        expected_samples=EXPECTED_SAMPLES,
        expected_pf_node=EXPECTED_PF_NODE,
        lattice_shape=LATTICE_SHAPE,
        pe_crit=PE_CRIT,
    )
    summary = run_postprocessing(
        conn,
        run_id=selected_run.run_id,
        make_plots=False,
        compute_risk_equivalence=True,
        risk_equivalence_pe_crit=PE_CRIT,
        lattice_shape=LATTICE_SHAPE,
    )

result = summary['risk_equivalence_result']
{
    'selected_run': selected_run.to_dict(),
    'risk_equivalence': result.to_dict(),
}

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from pbm.post.plot_style import plotting_context, save_figure

curve = result.numerical.curve
pe = np.asarray(curve.propagation_probability)
gap = np.asarray(curve.residual)
interval = np.asarray(curve.residual_confidence_interval)

with plotting_context(PLOT_MODE):
    fig, ax = plt.subplots()
    ax.plot(pe, gap, marker='o', label='PMF residual')
    ax.fill_between(pe, interval[:, 0], interval[:, 1], alpha=0.2,
                    label=f'{100 * result.numerical.confidence_level:.0f}% simultaneous MC interval')
    ax.axhline(0.0, color='black', linewidth=0.8)
    if result.numerical.estimate is not None:
        ax.axvline(result.numerical.estimate, linestyle='--',
                   label=rf"Numerical $p_e^\star={result.numerical.estimate:.4f}$")
    if result.analytical.estimate is not None:
        ax.axvline(result.analytical.estimate, linestyle=':',
                   label=rf"Analytical $p_e^\star={result.analytical.estimate:.4f}$")
    ax.set_xlabel(r'$p_e$ (fully coupled)')
    ax.set_ylabel(r'$P(N_f=1)-\sum_{n=2}^{N}(n-1)P(N_f=n)$')
    ax.grid(True)
    ax.legend()
    suffix = 'pdf' if PLOT_MODE == 'publication' else 'png'
    save_figure(fig, FIGS / f'risk_equivalence_gap.{suffix}', mode=PLOT_MODE, dpi=200)
    if PLOT_MODE == 'standard':
        plt.show()
    plt.close(fig)

A run created before `pf_node_provenance` was introduced is intentionally rejected; rerun the fully coupled cell in `percolation.ipynb` with the current code. The analytical result is labelled case-dependent in the structured output because its approximation has no guaranteed direction of bias for arbitrary finite topologies. If the numerical result reports no crossing, multiple crossings, unresolved Monte Carlo uncertainty, or a wide root bracket, refine the scan rather than substituting a closest grid point.